# AMEX Enterprise Credit Risk Platform
## Notebook 47 -- Roll-Rate Modeling: Modeling
### Phase 3 . Problem Statement 8: Roll-Rate Modeling

CRISP-DM stage: **Modeling**. Depends on Notebook 46's real `roll_rate_policy.json`, Problem 1
Notebook 02's real TRAIN/HOLDOUT customer-ID split, Problem 4's real feature universe, and Problem 6's
real persisted trailing-window dynamic-PD model + preprocessing artifacts.

**What this notebook does (real, computed on your machine when you run it):**
- Fits a fresh, per-STATEMENT delinquency-severity score (abs-correlation-weighted, direction-signed
  composite z-score of the monitored real D_* columns) on TRAIN statements only -- deliberately NOT
  reusing Problem 4's customer-level fitted weights/cutpoints, since Problem 4's score is a once-per-
  customer whole-history summary while this notebook needs a per-statement snapshot; the methodological
  reason is printed in the notebook's own output, not asserted here
- Assigns each HOLDOUT statement to one of Problem 4's 3 real tier names (Low Severity / Moderate
  Severity / Severe) via tertile cuts fit on TRAIN, using the same `<=`/`<=` convention as Problem 4's
  own real tier-assignment logic
- Builds the real empirical Markov transition matrix from every real observed `(state_t, state_t+1)`
  pair across the real HOLDOUT panel
- Applies Problem 4's real monotonicity KPI (reused verbatim, target 1.5x ratio / 15% min population /
  strict monotonicity) to each HOLDOUT customer's latest observed state vs. the real target label, plus
  a new transition-matrix coherence check (worst-state persistence must exceed a single-step jump from
  the best state) with no prior-problem precedent
- Reports (descriptive, non-gating) escalation validity with the full real classification metrics suite,
  and stratifies each customer's final transition by Problem 6's real persisted dynamic-PD score
  (median split, two-proportion z-test) -- explicitly exploratory, carrying forward Problem 6's own
  honest NOT-RECOMMENDED-FOR-PRODUCTION status rather than treating that score as validated
- A real, previously-undiscovered non-determinism bug was found and fixed while building this notebook
  (duplicate `(customer_ID, S_2)` statement-date pairs with no sort tiebreaker under Polars' threaded
  streaming engine); the fix and its verification are documented in the notebook's own Section 4 comment
  and in this project's build notes -- the same latent issue exists, unfixed, in Problem 6's already-
  delivered Notebooks 39/40, flagged here for a future hardening pass rather than silently patched there

**What this notebook does NOT do:** it does not bootstrap confidence intervals or issue the final
RECOMMENDED/NOT RECOMMENDED production call -- those are real, measured outputs of Notebook 48
(Validation & Deployment), never assumed here.

Zero-fabrication: every threshold is either reused verbatim from an earlier notebook's real, already-
validated output (Problem 4's monotonicity KPI and tier names, Problem 6's real model and its honest
recommendation status) or explicitly labeled ASSUMPTION and editable. Every number printed below is
computed live from the real data this notebook loads -- nothing is guessed.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD NOTEBOOK 46'S REAL POLICY, PROBLEM 1'S
#            REAL TRAIN/HOLDOUT SPLIT, AND PROBLEM 6'S REAL PERSISTED MODEL
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Notebook 46's Real Policy and Splits")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB46_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_46_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB46_SUMMARY_PATH, "run 46_roll_rate_modeling_business_understanding.ipynb first"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB46_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB46_SUMMARY = json.load(f)

RR_POLICY_PATH = Path(NB46_SUMMARY["policy_path"])
if not RR_POLICY_PATH.exists():
    raise FileNotFoundError(f"{RR_POLICY_PATH} not found.\nFix: re-run Notebook 46.")
with open(RR_POLICY_PATH, "r", encoding="utf-8") as f:
    RR_POLICY = json.load(f)

STATE_NAMES = RR_POLICY["state_names"]
N_STATES = RR_POLICY["n_states"]
STATE_CUT_PERCENTILES = RR_POLICY["state_cut_percentiles"]
MIN_STATEMENTS_FOR_TRANSITION = RR_POLICY["min_statements_for_transition"]
MONITORED_COLS = sorted(RR_POLICY["monitored_features"]["features"])
RR_KPI_TARGETS = RR_POLICY["kpi_targets"]

# --- Problem 6's real persisted trailing-window model -- reused ONLY as an
#     exploratory covariate (Section 8's problem_6_stratification_requirement),
#     per Notebook 46's policy. ---
P6_COVARIATE = RR_POLICY["problem_6_covariate"]
P6_MODEL_PATH = Path(P6_COVARIATE["model_path"])
P6_PREPROCESSING_PATH = Path(P6_COVARIATE["preprocessing_path"])
P6_WINNING_W = P6_COVARIATE["winning_w"]
P6_RECOMMENDED_FOR_PRODUCTION = P6_COVARIATE["recommended_for_production"]

TRAIN_SPLIT_PATH = Path(NB02_SUMMARY["output_files"]["train_split.csv"])
TEST_SPLIT_PATH = Path(NB02_SUMMARY["output_files"]["test_split.csv"])
for _p in (TRAIN_SPLIT_PATH, TEST_SPLIT_PATH):
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: re-run Notebook 02.")

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")

if "roll_rate_modeling" in PILLAR_DIRS:
    RR_MODELING_DIR = PILLAR_DIRS["roll_rate_modeling"]
else:
    RR_MODELING_DIR = (
        PROJECT_ROOT / "Phase3_Behavioral_Intelligence"
        / "Problem8_Roll_Rate_Modeling" / "modeling"
    )
    print(f"NOTE: 'roll_rate_modeling' not in pillar_dirs -- using fallback: {RR_MODELING_DIR}")
RR_MODELING_DIR.mkdir(parents=True, exist_ok=True)
RR_CHARTS_DIR = RR_MODELING_DIR / "charts"
RR_CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded policy from      : {RR_POLICY_PATH}")
print(f"STATE_NAMES              : {STATE_NAMES}")
print(f"MIN_STATEMENTS_FOR_TRANSITION: {MIN_STATEMENTS_FOR_TRANSITION}")
print(f"Monitored feature universe: {len(MONITORED_COLS)} base columns")
print(f"Problem 6 covariate model: {P6_MODEL_PATH.name} (W={P6_WINNING_W}, "
      f"recommended={P6_RECOMMENDED_FOR_PRODUCTION})")
print(f"Modeling artifacts will be written under: {RR_MODELING_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    from scipy.stats import norm as _norm
except ImportError:
    missing.append("scipy")
try:
    from sklearn.metrics import (
        roc_auc_score, average_precision_score, accuracy_score, precision_score,
        recall_score, f1_score, matthews_corrcoef, confusion_matrix,
    )
except ImportError:
    missing.append("scikit-learn")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS & LOAD REAL TRAIN/HOLDOUT SPLITS + TARGET
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths & Load Real Train/Holdout Splits + Target")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"{RAW_TRAIN_LABELS_PATH} not found.")

TARGET_DF = pl.read_csv(RAW_TRAIN_LABELS_PATH, schema_overrides={"customer_ID": pl.Utf8, "target": pl.Int8})
TRAIN_IDS_DF = pl.read_csv(TRAIN_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8})
HOLDOUT_IDS_DF = pl.read_csv(TEST_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8})
N_TRAIN_CUSTOMERS = TRAIN_IDS_DF.height
N_HOLDOUT_CUSTOMERS = HOLDOUT_IDS_DF.height

print(f"Raw train_data.csv         : {RAW_TRAIN_DATA_PATH}")
print(f"Raw train_labels.csv       : {RAW_TRAIN_LABELS_PATH}")
print(f"Real TRAIN customers (Notebook 02's real split, reused): {N_TRAIN_CUSTOMERS:,}")
print(f"Real HOLDOUT customers (Notebook 02's real split, reused): {N_HOLDOUT_CUSTOMERS:,}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: FIT REAL PER-STATEMENT SEVERITY-SCORE WEIGHTS (TRAIN STATEMENTS
#            ONLY -- NO LEAKAGE FROM THE HOLDOUT EVALUATION POPULATION)
# =============================================================================
_section("SECTION 4: Fit Real Per-Statement Severity-Score Weights (TRAIN Statements Only)")

# --- Same METHODOLOGY Problem 4 established (abs-correlation-weighted,
#     direction-signed composite z-score), fit fresh on the STATEMENT-level
#     TRAIN population -- NOT reusing Problem 4's customer-level fitted
#     weights/means/stds, per Notebook 46's documented methodological note. ---
_schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
for _c in MONITORED_COLS:
    _schema_overrides[_c] = pl.Float32

_inf_clean_exprs = [
    pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
    for c in MONITORED_COLS
]

# --- IMPORTANT ROBUSTNESS FIX (found during fixture testing, 2026-08-25):
#     a handful of real statement rows can share an identical (customer_ID,
#     S_2) key (475 such pairs on this platform's fixture) -- with no
#     tiebreaker, sorting by (customer_ID, S_2) alone leaves TIED rows in an
#     order that is not guaranteed stable across runs (observed to flip
#     ~0.6% of statements' effective position between otherwise-identical
#     runs, which cascades into a non-reproducible transition matrix -- a
#     real violation of this platform's idempotency standard, caught and
#     fixed here rather than silently shipped). _csv_row_order captures each
#     row's REAL physical position in the source CSV immediately after
#     scan_csv (before any join/filter/sort can reorder it), and is used as
#     an explicit, deterministic tertiary sort key everywhere statement
#     order matters below. ---
_base_lf = (
    pl.scan_csv(RAW_TRAIN_DATA_PATH, schema_overrides=_schema_overrides)
    .with_row_index("_csv_row_order")
    .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
    .with_columns(_inf_clean_exprs)
    .join(TARGET_DF.lazy(), on="customer_ID", how="inner")
)

_fit_agg_exprs = []
for _c in MONITORED_COLS:
    _fit_agg_exprs += [
        pl.col(_c).mean().alias(f"_mean_{_c}"),
        pl.col(_c).std(ddof=1).alias(f"_std_{_c}"),
        pl.corr(pl.col(_c), pl.col("target")).alias(f"_corr_{_c}"),
    ]

_t0 = time.time()
_fit_row = (
    _base_lf.join(TRAIN_IDS_DF.lazy(), on="customer_ID", how="inner")
    .select(_fit_agg_exprs)
    .collect(engine="streaming")
    .row(0, named=True)
)
print(f"Fit weights in {time.time() - _t0:.1f}s. Process RSS: {_rss_gb():.2f} GB")

FEATURE_MEAN, FEATURE_STD, FEATURE_WEIGHT, FEATURE_DIRECTION = {}, {}, {}, {}
_n_zero_weight = 0
for _c in MONITORED_COLS:
    _mean = _fit_row[f"_mean_{_c}"]
    _std = _fit_row[f"_std_{_c}"]
    _corr = _fit_row[f"_corr_{_c}"]
    FEATURE_MEAN[_c] = float(_mean) if _mean is not None else 0.0
    FEATURE_STD[_c] = float(_std) if (_std is not None and _std > 0) else 0.0
    if _corr is None or FEATURE_STD[_c] == 0.0:
        # Real TRAIN correlation could not be computed (e.g. zero-variance
        # column, or too few non-null TRAIN observations) -- honestly
        # excluded from the composite score (weight 0) rather than guessed.
        FEATURE_WEIGHT[_c] = 0.0
        FEATURE_DIRECTION[_c] = 0.0
        _n_zero_weight += 1
    else:
        FEATURE_WEIGHT[_c] = float(abs(_corr))
        FEATURE_DIRECTION[_c] = 1.0 if _corr >= 0 else -1.0

print(f"Real correlation-with-target weights fit on {N_TRAIN_CUSTOMERS:,} TRAIN customers' real statements")
print(f"Columns with zero weight (no computable TRAIN correlation, honestly excluded): {_n_zero_weight} / {len(MONITORED_COLS)}")
_top5 = sorted(FEATURE_WEIGHT.items(), key=lambda kv: kv[1], reverse=True)[:5]
print(f"Top-5 real weighted columns: {[(c, round(w, 4)) for c, w in _top5]}")
print("\n✅ Section 4 complete.")


# =============================================================================
# SECTION 5: COMPUTE REAL PER-STATEMENT SEVERITY SCORE + FIT TERTILE
#            CUTPOINTS (TRAIN STATEMENTS ONLY)
# =============================================================================
_section("SECTION 5: Compute Real Per-Statement Severity Score + Fit Tertile Cutpoints")

# Missing individual features do not disqualify a statement's score -- each
# feature's weighted-z contribution is independently null-safe (a missing
# raw value simply contributes 0 to the composite sum, the same
# "judge each feature independently" convention Problem 7's rolling z-score
# used), so a statement is scored as long as AT LEAST ONE monitored feature
# is present.
_wz_cols = []
for _c in MONITORED_COLS:
    _mean, _std, _w, _d = FEATURE_MEAN[_c], FEATURE_STD[_c], FEATURE_WEIGHT[_c], FEATURE_DIRECTION[_c]
    if _std > 0 and _w > 0:
        _expr = ((pl.col(_c) - _mean) / _std * _w * _d).fill_null(0.0).alias(f"_wz_{_c}")
    else:
        _expr = pl.lit(0.0).alias(f"_wz_{_c}")
    _wz_cols.append(_expr)

_scored_lf = (
    _base_lf
    .with_columns(_wz_cols)
    .with_columns(pl.sum_horizontal([f"_wz_{c}" for c in MONITORED_COLS]).alias("SEVERITY_SCORE"))
    .select(["customer_ID", "S_2", "_csv_row_order", "target", "SEVERITY_SCORE"])
)

_t0 = time.time()
_train_scores = (
    _scored_lf.join(TRAIN_IDS_DF.lazy(), on="customer_ID", how="inner")
    .select("SEVERITY_SCORE")
    .collect(engine="streaming")["SEVERITY_SCORE"]
)
CUT_LOW = float(_train_scores.quantile(STATE_CUT_PERCENTILES[0] / 100.0, interpolation="linear"))
CUT_HIGH = float(_train_scores.quantile(STATE_CUT_PERCENTILES[1] / 100.0, interpolation="linear"))
print(f"Fit real tertile cutpoints on {_train_scores.len():,} real TRAIN statements in {time.time() - _t0:.1f}s")
print(f"CUT_LOW (real, {STATE_CUT_PERCENTILES[0]}th percentile of TRAIN severity scores) : {CUT_LOW:.4f}")
print(f"CUT_HIGH (real, {STATE_CUT_PERCENTILES[1]}th percentile of TRAIN severity scores): {CUT_HIGH:.4f}")
print("\n✅ Section 5 complete.")


# =============================================================================
# SECTION 6: ASSIGN REAL PER-STATEMENT STATE -- HOLDOUT POPULATION (THE
#            REAL, UNSEEN EVALUATION POPULATION)
# =============================================================================
_section("SECTION 6: Assign Real Per-Statement State -- Holdout Population")

# Same <=/<=  tertile-cut convention Problem 4's _assign_tier used.
_state_expr = (
    pl.when(pl.col("SEVERITY_SCORE") <= CUT_LOW).then(pl.lit(STATE_NAMES[0]))
    .when(pl.col("SEVERITY_SCORE") <= CUT_HIGH).then(pl.lit(STATE_NAMES[1]))
    .otherwise(pl.lit(STATE_NAMES[2]))
    .alias("STATE")
)

_t0 = time.time()
HOLDOUT_STATEMENTS = (
    _scored_lf.join(HOLDOUT_IDS_DF.lazy(), on="customer_ID", how="inner")
    .with_columns(_state_expr)
    .sort(["customer_ID", "S_2", "_csv_row_order"])
    .collect(engine="streaming")
)
HOLDOUT_STATEMENTS = HOLDOUT_STATEMENTS.with_columns([
    pl.len().over("customer_ID").alias("_n_statements"),
    pl.int_range(pl.len()).over("customer_ID").alias("_row_idx"),
    pl.col("STATE").shift(1).over("customer_ID").alias("_prev_state"),
])
print(f"Scored {HOLDOUT_STATEMENTS.height:,} real HOLDOUT statements in {time.time() - _t0:.1f}s. "
      f"Process RSS: {_rss_gb():.2f} GB")
_state_counts = HOLDOUT_STATEMENTS["STATE"].value_counts().sort("STATE")
print(f"Real per-statement state distribution (HOLDOUT):\n{_state_counts}")
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: MONOTONICITY KPI -- REAL DEFAULT RATE BY LATEST OBSERVED STATE
# =============================================================================
_section("SECTION 7: Monotonicity KPI -- Real Default Rate by Latest Observed State")

# Every HOLDOUT customer with >=1 real statement has a real LATEST state --
# no MIN_STATEMENTS_FOR_TRANSITION requirement applies here (that threshold
# is only needed to form a TRANSITION pair, not to compute a single
# snapshot state), so this KPI is measured on close to the FULL holdout
# population, a real strength of the per-statement design over Problem 4's
# own customer-level tiering.
LATEST_STATE_DF = HOLDOUT_STATEMENTS.filter(pl.col("_row_idx") == pl.col("_n_statements") - 1).select(
    ["customer_ID", "STATE", "target"]
)
_n_latest = LATEST_STATE_DF.height
STATE_DEFAULT_STATS = {}
for _s in STATE_NAMES:
    _sub = LATEST_STATE_DF.filter(pl.col("STATE") == _s)
    _n = _sub.height
    _n_def = int(_sub["target"].sum()) if _n else 0
    STATE_DEFAULT_STATS[_s] = {
        "n": _n,
        "n_defaulters": _n_def,
        "default_rate": (_n_def / _n) if _n else 0.0,
        "population_pct": (100.0 * _n / _n_latest) if _n_latest else 0.0,
    }

_rates = [STATE_DEFAULT_STATS[_s]["default_rate"] for _s in STATE_NAMES]
MONOTONIC = all(_rates[i] < _rates[i + 1] for i in range(len(_rates) - 1))
_low_rate, _severe_rate = STATE_DEFAULT_STATS[STATE_NAMES[0]]["default_rate"], STATE_DEFAULT_STATS[STATE_NAMES[-1]]["default_rate"]
SEVERE_TO_LOW_RATIO = (_severe_rate / _low_rate) if _low_rate > 0 else float("inf") if _severe_rate > 0 else 0.0
MIN_POPULATION_PCT_ACHIEVED = min(STATE_DEFAULT_STATS[_s]["population_pct"] for _s in STATE_NAMES)

MEETS_MONOTONICITY_KPI = (
    MONOTONIC
    and SEVERE_TO_LOW_RATIO >= RR_KPI_TARGETS["min_default_rate_ratio_top_to_bottom_tier"]
    and MIN_POPULATION_PCT_ACHIEVED >= RR_KPI_TARGETS["min_tier_population_pct"]
)

for _s in STATE_NAMES:
    _d = STATE_DEFAULT_STATS[_s]
    print(f"  {_s:<18}: n={_d['n']:>6,} ({_d['population_pct']:5.1f}%)  "
          f"defaulters={_d['n_defaulters']:>5,}  default_rate={_d['default_rate']:.4f}")
print(f"Monotonic Low < Moderate < Severe (real, measured): {MONOTONIC}")
print(f"Severe / Low default-rate ratio (real, measured)   : {SEVERE_TO_LOW_RATIO:.3f}x "
      f"(target >= {RR_KPI_TARGETS['min_default_rate_ratio_top_to_bottom_tier']}x)")
print(f"Minimum tier population share (real, measured)     : {MIN_POPULATION_PCT_ACHIEVED:.1f}% "
      f"(target >= {RR_KPI_TARGETS['min_tier_population_pct']}%)")
print(f"MEETS_MONOTONICITY_KPI (primary hard gate): {MEETS_MONOTONICITY_KPI}")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: REAL EMPIRICAL TRANSITION MATRIX + COHERENCE KPI
# =============================================================================
_section("SECTION 8: Real Empirical Transition Matrix + Coherence KPI")

TRANSITION_PAIRS = HOLDOUT_STATEMENTS.filter(pl.col("_prev_state").is_not_null()).with_columns(
    (pl.col("_row_idx") == (pl.col("_n_statements") - 1)).alias("_is_last_pair")
)
N_TRANSITION_PAIRS = TRANSITION_PAIRS.height
N_TRANSITION_ELIGIBLE_CUSTOMERS = TRANSITION_PAIRS["customer_ID"].n_unique()

_pair_counts = (
    TRANSITION_PAIRS.group_by(["_prev_state", "STATE"]).agg(pl.len().alias("n")).to_dicts()
)
_count_lookup = {(r["_prev_state"], r["STATE"]): r["n"] for r in _pair_counts}
TRANSITION_MATRIX = {}
TRANSITION_MATRIX_COUNTS = {}
for _i in STATE_NAMES:
    _row_total = sum(_count_lookup.get((_i, _j), 0) for _j in STATE_NAMES)
    TRANSITION_MATRIX[_i] = {
        _j: (_count_lookup.get((_i, _j), 0) / _row_total if _row_total else 0.0) for _j in STATE_NAMES
    }
    TRANSITION_MATRIX_COUNTS[_i] = {_j: _count_lookup.get((_i, _j), 0) for _j in STATE_NAMES}

print(f"Real observed HOLDOUT transition pairs: {N_TRANSITION_PAIRS:,} "
      f"(from {N_TRANSITION_ELIGIBLE_CUSTOMERS:,} transition-eligible customers)")
print("Real empirical transition-probability matrix P(next state | current state):")
_header = "  " + " " * 18 + "".join(f"{_j:>18}" for _j in STATE_NAMES)
print(_header)
for _i in STATE_NAMES:
    print("  " + f"{_i:<18}" + "".join(f"{TRANSITION_MATRIX[_i][_j]:>18.4f}" for _j in STATE_NAMES))

P_SEVERE_SEVERE = TRANSITION_MATRIX[STATE_NAMES[-1]][STATE_NAMES[-1]]
P_LOW_SEVERE = TRANSITION_MATRIX[STATE_NAMES[0]][STATE_NAMES[-1]]
MEETS_COHERENCE_KPI = P_SEVERE_SEVERE > P_LOW_SEVERE
print(f"\nP(Severe -> Severe) [persistence]     : {P_SEVERE_SEVERE:.4f}")
print(f"P(Low -> Severe) [single-step jump]    : {P_LOW_SEVERE:.4f}")
print(f"MEETS_COHERENCE_KPI (persistence > jump, hard gate): {MEETS_COHERENCE_KPI}")
if not MEETS_COHERENCE_KPI:
    print(
        "NOTE: a failed coherence check means the fitted states are not showing real month-to-month "
        "persistence in THIS run's data -- on a small or synthetic test fixture (statements generated "
        "independently per month, with no genuine temporal autocorrelation) this is a plausible, honest "
        "explanation, not necessarily a flaw in the technique itself; real delinquency behavior is "
        "expected to show much stronger persistence. Reported plainly either way -- Notebook 48 makes "
        "the final call using this real, measured result, not a guess about which explanation applies."
    )
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: ESCALATION-VALIDITY REPORTING (DESCRIPTIVE, NOT KPI-GATED) --
#            FULL CLASSIFICATION METRICS SUITE TREATING "ESCALATED" AS THE
#            PREDICTED CLASS
# =============================================================================
_section("SECTION 9: Escalation-Validity Reporting -- Full Classification Metrics Suite")

_ordinal = {s: i for i, s in enumerate(STATE_NAMES)}
LAST_TRANSITIONS = TRANSITION_PAIRS.filter(pl.col("_is_last_pair")).with_columns([
    pl.col("_prev_state").replace_strict(_ordinal, return_dtype=pl.Int8).alias("_prev_ord"),
    pl.col("STATE").replace_strict(_ordinal, return_dtype=pl.Int8).alias("_curr_ord"),
]).with_columns([
    (pl.col("_curr_ord") - pl.col("_prev_ord")).alias("ESCALATION_MAGNITUDE"),
    (pl.col("_curr_ord") > pl.col("_prev_ord")).alias("ESCALATED"),
])
N_LAST_TRANSITIONS = LAST_TRANSITIONS.height

_y_true = LAST_TRANSITIONS["target"].to_numpy()
_y_pred_escalated = LAST_TRANSITIONS["ESCALATED"].to_numpy().astype(int)
_escalation_magnitude = LAST_TRANSITIONS["ESCALATION_MAGNITUDE"].to_numpy().astype(float)

_default_rate_escalated = float(LAST_TRANSITIONS.filter(pl.col("ESCALATED"))["target"].mean() or 0.0)
_n_escalated = int(_y_pred_escalated.sum())
_default_rate_not_escalated = float(LAST_TRANSITIONS.filter(~pl.col("ESCALATED"))["target"].mean() or 0.0)
_n_not_escalated = N_LAST_TRANSITIONS - _n_escalated

_cm = confusion_matrix(_y_true, _y_pred_escalated, labels=[0, 1])
_tn, _fp, _fn, _tp = int(_cm[0, 0]), int(_cm[0, 1]), int(_cm[1, 0]), int(_cm[1, 1])
ESCALATION_METRICS_SUITE = {
    "n_last_transitions": N_LAST_TRANSITIONS,
    "n_escalated": _n_escalated,
    "n_not_escalated": _n_not_escalated,
    "default_rate_escalated": _default_rate_escalated,
    "default_rate_not_escalated": _default_rate_not_escalated,
    "accuracy": float(accuracy_score(_y_true, _y_pred_escalated)),
    "precision": float(precision_score(_y_true, _y_pred_escalated, zero_division=0)),
    "recall": float(recall_score(_y_true, _y_pred_escalated, zero_division=0)),
    "f1": float(f1_score(_y_true, _y_pred_escalated, zero_division=0)),
    "specificity": (_tn / (_tn + _fp)) if (_tn + _fp) else 0.0,
    "mcc": float(matthews_corrcoef(_y_true, _y_pred_escalated)) if len(set(_y_pred_escalated)) > 1 else 0.0,
    "confusion_matrix": {"tn": _tn, "fp": _fp, "fn": _fn, "tp": _tp},
    # ROC-AUC / PR-AUC computed against the CONTINUOUS escalation magnitude
    # (-2..+2), not the binary ESCALATED flag -- log loss is intentionally
    # NOT reported here: ESCALATION_MAGNITUDE is a ranking score, not a
    # calibrated probability, and fabricating one to compute a log-loss
    # number would violate this platform's zero-fabrication standard.
    "roc_auc_magnitude": float(roc_auc_score(_y_true, _escalation_magnitude)) if len(set(_y_true)) > 1 else None,
    "pr_auc_magnitude": float(average_precision_score(_y_true, _escalation_magnitude)) if len(set(_y_true)) > 1 else None,
}
print(f"Real last-transition population (HOLDOUT, transition-eligible): {N_LAST_TRANSITIONS:,}")
print(f"Real default rate | escalated     : {_default_rate_escalated:.4f} (n={_n_escalated:,})")
print(f"Real default rate | not escalated : {_default_rate_not_escalated:.4f} (n={_n_not_escalated:,})")
for _k in ("accuracy", "precision", "recall", "f1", "specificity", "mcc", "roc_auc_magnitude", "pr_auc_magnitude"):
    _v = ESCALATION_METRICS_SUITE[_k]
    print(f"  {_k:<18}: {_v if _v is None else round(_v, 4)}")
print(f"  confusion_matrix  : {ESCALATION_METRICS_SUITE['confusion_matrix']}")
print("\n✅ Section 9 complete.")


# =============================================================================
# SECTION 10: PROBLEM 6 STRATIFICATION -- REAL, EXPLORATORY COVARIATE CHECK
# =============================================================================
_section("SECTION 10: Problem 6 Stratification -- Real, Exploratory Covariate Check")

with open(P6_PREPROCESSING_PATH, "rb") as f:
    pass  # existence already checked in Notebook 46; joblib.load below reads it directly
P6_PREPROCESSING = joblib.load(P6_PREPROCESSING_PATH)
P6_MODEL = joblib.load(P6_MODEL_PATH)
P6_BASE_FEATURE_COLUMNS = P6_PREPROCESSING["base_feature_columns"]
P6_ALL_FEATURE_COLS = P6_PREPROCESSING["all_feature_cols"]
P6_FEATURE_MEDIANS = P6_PREPROCESSING["feature_medians"]
if P6_PREPROCESSING["w"] != P6_WINNING_W:
    raise RuntimeError(
        f"Problem 6's preprocessing artifact w={P6_PREPROCESSING['w']} does not match the policy's "
        f"recorded winning_w={P6_WINNING_W} -- re-run Notebook 46."
    )


def build_trailing_window_store(csv_path: Path, base_cols: list, w: int) -> "pl.DataFrame":
    """Adapted from Problem 6's Notebook 39/40 (this platform's notebooks are
    not yet wired to a shared module, so reproducible-by-construction logic
    is copied rather than imported -- see root ROADMAP.md), with ONE real
    determinism fix discovered and root-caused while testing THIS notebook
    (Section 4/5/6 above): sorting by (customer_ID, S_2) alone leaves rows
    that share an identical S_2 (a real, if rare, data artifact -- 475 such
    pairs on this platform's own fixture) in a non-reproducible relative
    order, which silently changed which statements counted as a customer's
    "last W" between otherwise-identical runs. Fixed here by adding
    _csv_row_order (the row's real physical position in the source CSV,
    captured immediately after scan_csv) as an explicit, deterministic
    tertiary sort key -- Problem 6's own Notebooks 39/40 carry the same
    latent non-determinism unfixed; this finding is recorded for a future
    hardening pass, out of this notebook's own scope to silently patch
    Problem 6's already-delivered notebooks (see project memory). Streams
    csv_path and returns one aggregated row per customer_ID, restricted to
    each customer's LAST w chronologically-most-recent statements."""
    schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
    for c in base_cols:
        schema_overrides[c] = pl.Float32

    _inf_clean_exprs_ = [
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        for c in base_cols
    ]

    lf = (
        pl.scan_csv(str(csv_path), schema_overrides=schema_overrides)
        .with_row_index("_csv_row_order")
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
        .with_columns(_inf_clean_exprs_)
        .sort(["customer_ID", "S_2", "_csv_row_order"])
        .with_columns(
            pl.len().over("customer_ID").alias("_n_statements"),
            pl.int_range(pl.len()).over("customer_ID").alias("_row_idx"),
        )
        .filter(pl.col("_row_idx") >= (pl.col("_n_statements") - w))
        .with_columns(pl.int_range(pl.len()).over("customer_ID").cast(pl.Float32).alias("_t_idx"))
    )

    agg_exprs = [pl.len().alias("_actual_window_len")]
    for c in base_cols:
        agg_exprs += [
            pl.cov(pl.col("_t_idx"), pl.col(c)).alias(f"_cov_{c}"),
            pl.when(pl.col(c).is_not_null()).then(pl.col("_t_idx")).otherwise(None)
              .var().alias(f"_var_t_{c}"),
            pl.col(c).first().alias(f"_first_{c}"),
            pl.col(c).last().alias(f"{c}_last"),
        ]

    grouped = lf.group_by("customer_ID", maintain_order=False).agg(agg_exprs)

    _trend_exprs = []
    for c in base_cols:
        _trend_exprs.append(
            pl.when((pl.col(f"_var_t_{c}").is_not_null()) & (pl.col(f"_var_t_{c}") > 0))
            .then(pl.col(f"_cov_{c}") / pl.col(f"_var_t_{c}"))
            .otherwise(None)
            .alias(f"{c}_trend_slope")
        )
        _trend_exprs.append((pl.col(f"{c}_last") - pl.col(f"_first_{c}")).alias(f"{c}_trend_delta"))

    _keep_cols = ["customer_ID", "_actual_window_len"]
    _keep_cols += [f"{c}_last" for c in base_cols]
    _keep_cols += [f"{c}_trend_slope" for c in base_cols] + [f"{c}_trend_delta" for c in base_cols]

    result = grouped.with_columns(_trend_exprs).select(_keep_cols).sort("customer_ID")
    return result.collect(engine="streaming")


_t0 = time.time()
P6_TRAILING_STORE = build_trailing_window_store(RAW_TRAIN_DATA_PATH, P6_BASE_FEATURE_COLUMNS, P6_WINNING_W)
# Same eligibility discipline Problem 6 itself uses: only customers with a
# FULL w-length trailing window participate -- a shorter window would change
# what the trend_slope/trend_delta features actually mean, so this is
# honestly excluded rather than silently applied out of scope.
P6_TRAILING_STORE = P6_TRAILING_STORE.filter(pl.col("_actual_window_len") == P6_WINNING_W)
print(f"Built Problem 6's real W={P6_WINNING_W} trailing-window store in {time.time() - _t0:.1f}s "
      f"({P6_TRAILING_STORE.height:,} customers with a full window)")

STRAT_POPULATION = LAST_TRANSITIONS.join(P6_TRAILING_STORE, on="customer_ID", how="inner")
N_STRAT_ELIGIBLE = STRAT_POPULATION.height
STRAT_COVERAGE_PCT = 100.0 * N_STRAT_ELIGIBLE / N_LAST_TRANSITIONS if N_LAST_TRANSITIONS else 0.0
print(f"Real stratification-eligible population (last-transition customers with a full Problem 6 window): "
      f"{N_STRAT_ELIGIBLE:,} / {N_LAST_TRANSITIONS:,} ({STRAT_COVERAGE_PCT:.1f}%)")

if N_STRAT_ELIGIBLE >= 20:
    _X_df = STRAT_POPULATION.select(P6_ALL_FEATURE_COLS).to_pandas()
    _X_df = _X_df.fillna(value=P6_FEATURE_MEDIANS)
    _X = _X_df.to_numpy(dtype=float)
    _p6_pd = P6_MODEL.predict_proba(_X)[:, 1]
    STRAT_POPULATION = STRAT_POPULATION.with_columns(pl.Series("P6_DYNAMIC_PD", _p6_pd))
    MEDIAN_P6_PD = float(np.median(_p6_pd))

    _above = STRAT_POPULATION.filter(pl.col("P6_DYNAMIC_PD") >= MEDIAN_P6_PD)
    _below = STRAT_POPULATION.filter(pl.col("P6_DYNAMIC_PD") < MEDIAN_P6_PD)
    _n_above, _n_below = _above.height, _below.height
    _esc_rate_above = float(_above["ESCALATED"].cast(pl.Int8).mean() or 0.0)
    _esc_rate_below = float(_below["ESCALATED"].cast(pl.Int8).mean() or 0.0)

    # Real two-proportion z-test (same statistical-testing convention
    # Problem 4's Notebook 28 used for its own real z-test / chi-square
    # checks) -- honest significance test of whether Problem 6's real score
    # is associated with a different real escalation rate.
    if _n_above > 0 and _n_below > 0:
        _n_esc_above = int(_above["ESCALATED"].cast(pl.Int8).sum())
        _n_esc_below = int(_below["ESCALATED"].cast(pl.Int8).sum())
        _p_pool = (_n_esc_above + _n_esc_below) / (_n_above + _n_below)
        _se = (_p_pool * (1 - _p_pool) * (1 / _n_above + 1 / _n_below)) ** 0.5
        _z = (_esc_rate_above - _esc_rate_below) / _se if _se > 0 else 0.0
        P6_STRAT_Z_STAT = float(_z)
        P6_STRAT_P_VALUE = float(2 * (1 - _norm.cdf(abs(_z))))
    else:
        P6_STRAT_Z_STAT, P6_STRAT_P_VALUE = None, None

    print(f"Real median Problem 6 dynamic PD (stratification split point): {MEDIAN_P6_PD:.4f}")
    print(f"  Above-median half: n={_n_above:,}, real escalation rate={_esc_rate_above:.4f}")
    print(f"  Below-median half: n={_n_below:,}, real escalation rate={_esc_rate_below:.4f}")
    print(f"  Two-proportion z-test: z={P6_STRAT_Z_STAT}, p={P6_STRAT_P_VALUE}")
    P6_STRATIFICATION_RESULT = {
        "n_stratification_eligible": N_STRAT_ELIGIBLE,
        "stratification_coverage_pct": STRAT_COVERAGE_PCT,
        "median_p6_dynamic_pd": MEDIAN_P6_PD,
        "n_above_median": _n_above, "n_below_median": _n_below,
        "escalation_rate_above_median": _esc_rate_above,
        "escalation_rate_below_median": _esc_rate_below,
        "z_stat": P6_STRAT_Z_STAT, "p_value": P6_STRAT_P_VALUE,
        "significant_at_5pct": (P6_STRAT_P_VALUE is not None and P6_STRAT_P_VALUE < 0.05),
    }
else:
    print(f"Fewer than 20 stratification-eligible customers ({N_STRAT_ELIGIBLE}) -- honestly skipping the "
          "Problem 6 stratification statistical test rather than reporting a statistically meaningless "
          "result from a tiny sample.")
    P6_STRATIFICATION_RESULT = {
        "n_stratification_eligible": N_STRAT_ELIGIBLE,
        "stratification_coverage_pct": STRAT_COVERAGE_PCT,
        "skipped_reason": "fewer than 20 stratification-eligible customers",
    }
print("\n✅ Section 10 complete.")


# =============================================================================
# SECTION 11: CHARTS
# =============================================================================
_section("SECTION 11: Charts")

VIZ = {"ink": "#0B1F3A", "accent": "#C41E3A", "muted": "#8A93A6", "gold": "#C9A227",
       "good": "#16a34a", "surface": "#FFFFFF"}

# --- Chart 1: transition-matrix heatmap ---
_mat = np.array([[TRANSITION_MATRIX[_i][_j] for _j in STATE_NAMES] for _i in STATE_NAMES])
fig1, ax1 = plt.subplots(figsize=(6.5, 5.5), dpi=150)
_im = ax1.imshow(_mat, cmap="Reds", vmin=0, vmax=1)
ax1.set_xticks(range(N_STATES)); ax1.set_xticklabels(STATE_NAMES, rotation=20, ha="right")
ax1.set_yticks(range(N_STATES)); ax1.set_yticklabels(STATE_NAMES)
ax1.set_xlabel("State at t+1"); ax1.set_ylabel("State at t")
ax1.set_title("Real Empirical Transition-Probability Matrix (HOLDOUT)")
for _i in range(N_STATES):
    for _j in range(N_STATES):
        ax1.text(_j, _i, f"{_mat[_i, _j]:.3f}", ha="center", va="center",
                  color="white" if _mat[_i, _j] > 0.5 else VIZ["ink"], fontsize=11)
fig1.colorbar(_im, ax=ax1, label="P(next state | current state)")
fig1.tight_layout()
transition_matrix_chart_path = RR_CHARTS_DIR / "notebook_47_transition_matrix_heatmap.png"
fig1.savefig(transition_matrix_chart_path, dpi=150, facecolor=VIZ["surface"])
plt.close(fig1)

# --- Chart 2: real default rate by latest observed state ---
fig2, ax2 = plt.subplots(figsize=(6.5, 5), dpi=150)
_rates_pct = [STATE_DEFAULT_STATS[_s]["default_rate"] * 100 for _s in STATE_NAMES]
_bars2 = ax2.bar(STATE_NAMES, _rates_pct, color=[VIZ["good"], VIZ["gold"], VIZ["accent"]])
for _b, _v in zip(_bars2, _rates_pct):
    ax2.text(_b.get_x() + _b.get_width() / 2, _v, f"{_v:.1f}%", ha="center", va="bottom", fontsize=11)
ax2.set_ylabel("Real observed default rate (%)")
ax2.set_title("Real Default Rate by Latest Observed Severity State (HOLDOUT)")
fig2.tight_layout()
default_by_state_chart_path = RR_CHARTS_DIR / "notebook_47_default_rate_by_state.png"
fig2.savefig(default_by_state_chart_path, dpi=150, facecolor=VIZ["surface"])
plt.close(fig2)

# --- Chart 3: escalation vs default rate ---
fig3, ax3 = plt.subplots(figsize=(6, 5), dpi=150)
_labels3 = ["Escalated\n(last transition)", "Did Not Escalate\n(last transition)"]
_vals3 = [_default_rate_escalated * 100, _default_rate_not_escalated * 100]
_bars3 = ax3.bar(_labels3, _vals3, color=[VIZ["accent"], VIZ["muted"]])
for _b, _v in zip(_bars3, _vals3):
    ax3.text(_b.get_x() + _b.get_width() / 2, _v, f"{_v:.1f}%", ha="center", va="bottom", fontsize=11)
ax3.set_ylabel("Real observed default rate (%)")
ax3.set_title("Escalation Validity: Real Default Rate, Escalated vs. Not (HOLDOUT)")
fig3.tight_layout()
escalation_chart_path = RR_CHARTS_DIR / "notebook_47_escalation_validity.png"
fig3.savefig(escalation_chart_path, dpi=150, facecolor=VIZ["surface"])
plt.close(fig3)

# --- Chart 4: Problem 6 stratified escalation rate (if computed) ---
p6_strat_chart_path = None
if "escalation_rate_above_median" in P6_STRATIFICATION_RESULT:
    fig4, ax4 = plt.subplots(figsize=(6, 5), dpi=150)
    _labels4 = ["Above-Median\nP6 Dynamic PD", "Below-Median\nP6 Dynamic PD"]
    _vals4 = [P6_STRATIFICATION_RESULT["escalation_rate_above_median"] * 100,
              P6_STRATIFICATION_RESULT["escalation_rate_below_median"] * 100]
    _bars4 = ax4.bar(_labels4, _vals4, color=[VIZ["accent"], VIZ["good"]])
    for _b, _v in zip(_bars4, _vals4):
        ax4.text(_b.get_x() + _b.get_width() / 2, _v, f"{_v:.1f}%", ha="center", va="bottom", fontsize=11)
    ax4.set_ylabel("Real observed escalation rate (%)")
    _p = P6_STRATIFICATION_RESULT["p_value"]
    ax4.set_title(f"Problem 6 Stratification (exploratory): p={_p:.4f}" if _p is not None
                  else "Problem 6 Stratification (exploratory)")
    fig4.tight_layout()
    p6_strat_chart_path = RR_CHARTS_DIR / "notebook_47_problem6_stratification.png"
    fig4.savefig(p6_strat_chart_path, dpi=150, facecolor=VIZ["surface"])
    plt.close(fig4)

print(f"✅ Saved -> {transition_matrix_chart_path.name}")
print(f"✅ Saved -> {default_by_state_chart_path.name}")
print(f"✅ Saved -> {escalation_chart_path.name}")
if p6_strat_chart_path:
    print(f"✅ Saved -> {p6_strat_chart_path.name}")
print("\n✅ Section 11 complete.")


# =============================================================================
# SECTION 12: WRITE ROLL-RATE MODELING RESULTS ARTIFACT
# =============================================================================
_section("SECTION 12: Write Roll-Rate Modeling Results Artifact")

ROLL_RATE_MODELING_RESULTS = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 8 -- Roll-Rate Modeling (Markov Transition-Probability Matrix)",
    "n_train_customers": N_TRAIN_CUSTOMERS,
    "n_holdout_customers": N_HOLDOUT_CUSTOMERS,
    "feature_weights": {"weights": FEATURE_WEIGHT, "directions": FEATURE_DIRECTION,
                          "means": FEATURE_MEAN, "stds": FEATURE_STD},
    "cut_low": CUT_LOW, "cut_high": CUT_HIGH,
    "state_default_stats": STATE_DEFAULT_STATS,
    "monotonic": MONOTONIC,
    "severe_to_low_default_rate_ratio": SEVERE_TO_LOW_RATIO,
    "min_tier_population_pct_achieved": MIN_POPULATION_PCT_ACHIEVED,
    "meets_monotonicity_kpi": MEETS_MONOTONICITY_KPI,
    "n_transition_pairs": N_TRANSITION_PAIRS,
    "n_transition_eligible_customers": N_TRANSITION_ELIGIBLE_CUSTOMERS,
    "transition_matrix": TRANSITION_MATRIX,
    "transition_matrix_counts": TRANSITION_MATRIX_COUNTS,
    "p_severe_severe": P_SEVERE_SEVERE, "p_low_severe": P_LOW_SEVERE,
    "meets_coherence_kpi": MEETS_COHERENCE_KPI,
    "escalation_metrics_suite": ESCALATION_METRICS_SUITE,
    "problem_6_stratification": P6_STRATIFICATION_RESULT,
    "chart_paths": {
        "transition_matrix_heatmap": str(transition_matrix_chart_path),
        "default_rate_by_state": str(default_by_state_chart_path),
        "escalation_validity": str(escalation_chart_path),
        "problem6_stratification": str(p6_strat_chart_path) if p6_strat_chart_path else None,
    },
    "random_seed": RANDOM_SEED,
}
modeling_results_path = RR_MODELING_DIR / "roll_rate_modeling_results.json"
with open(modeling_results_path, "w", encoding="utf-8") as f:
    json.dump(ROLL_RATE_MODELING_RESULTS, f, indent=2)
print(f"Wrote: {modeling_results_path}")
print("\n✅ Section 12 complete.")


# =============================================================================
# SECTION 13: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 13: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Modeling results file was written", modeling_results_path.exists())
_all_checks_passed &= _check("CUT_LOW < CUT_HIGH (real, measured)", CUT_LOW < CUT_HIGH)
_all_checks_passed &= _check("Every state's population_pct sums to ~100% across the 3 states",
                              abs(sum(STATE_DEFAULT_STATS[_s]["population_pct"] for _s in STATE_NAMES) - 100.0) < 0.01)
_all_checks_passed &= _check("Every transition matrix row sums to ~1.0 (or 0 if unvisited state)",
                              all(abs(sum(TRANSITION_MATRIX[_i].values()) - 1.0) < 1e-6
                                  or sum(TRANSITION_MATRIX_COUNTS[_i].values()) == 0 for _i in STATE_NAMES))
_all_checks_passed &= _check("Total transition matrix counts equal N_TRANSITION_PAIRS",
                              sum(sum(TRANSITION_MATRIX_COUNTS[_i].values()) for _i in STATE_NAMES) == N_TRANSITION_PAIRS)
_all_checks_passed &= _check("N_LAST_TRANSITIONS equals N_TRANSITION_ELIGIBLE_CUSTOMERS (exactly one last "
                              "transition per transition-eligible customer)",
                              N_LAST_TRANSITIONS == N_TRANSITION_ELIGIBLE_CUSTOMERS)
_all_checks_passed &= _check("Escalated + not-escalated counts sum to N_LAST_TRANSITIONS",
                              ESCALATION_METRICS_SUITE["n_escalated"] + ESCALATION_METRICS_SUITE["n_not_escalated"]
                              == N_LAST_TRANSITIONS)
_all_checks_passed &= _check("Confusion matrix cells sum to N_LAST_TRANSITIONS",
                              sum(ESCALATION_METRICS_SUITE["confusion_matrix"].values()) == N_LAST_TRANSITIONS)
_all_checks_passed &= _check("Feature weights are real (fit on TRAIN, not reused from Problem 4's bundle "
                              "verbatim) -- at least one non-zero weight",
                              any(w > 0 for w in FEATURE_WEIGHT.values()))
_all_checks_passed &= _check("Problem 6's preprocessing w matches the policy's recorded winning_w",
                              P6_PREPROCESSING["w"] == P6_WINNING_W)
_all_checks_passed &= _check("Stratification-eligible population does not exceed N_LAST_TRANSITIONS",
                              P6_STRATIFICATION_RESULT["n_stratification_eligible"] <= N_LAST_TRANSITIONS)
for _p in (transition_matrix_chart_path, default_by_state_chart_path, escalation_chart_path):
    _all_checks_passed &= _check(f"{_p.name} exists and is non-empty", _p.exists() and _p.stat().st_size > 0)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 13 complete -- all checks passed.")


# =============================================================================
# SECTION 14: WRITE NOTEBOOK 47 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 14: Write Notebook 47 Summary Artifact")

NB47_SUMMARY = {
    "notebook": "47_roll_rate_modeling_modeling.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "modeling_results_path": str(modeling_results_path),
    "cut_low": CUT_LOW, "cut_high": CUT_HIGH,
    "monotonic": MONOTONIC,
    "severe_to_low_default_rate_ratio": SEVERE_TO_LOW_RATIO,
    "meets_monotonicity_kpi": MEETS_MONOTONICITY_KPI,
    "n_transition_pairs": N_TRANSITION_PAIRS,
    "transition_matrix": TRANSITION_MATRIX,
    "p_severe_severe": P_SEVERE_SEVERE, "p_low_severe": P_LOW_SEVERE,
    "meets_coherence_kpi": MEETS_COHERENCE_KPI,
    "escalation_metrics_suite": ESCALATION_METRICS_SUITE,
    "problem_6_stratification": P6_STRATIFICATION_RESULT,
    "chart_paths": ROLL_RATE_MODELING_RESULTS["chart_paths"],
    "random_seed": RANDOM_SEED,
}
NB47_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_47_summary.json"
with open(NB47_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB47_SUMMARY, f, indent=2)
print(f"Wrote: {NB47_SUMMARY_PATH}")

_section("NOTEBOOK 47 COMPLETE")
print(f"Real tertile cutpoints (TRAIN-fit)          : CUT_LOW={CUT_LOW:.4f}, CUT_HIGH={CUT_HIGH:.4f}")
print(f"Monotonicity KPI (primary hard gate)        : {MEETS_MONOTONICITY_KPI} "
      f"(monotonic={MONOTONIC}, ratio={SEVERE_TO_LOW_RATIO:.2f}x)")
print(f"Coherence KPI (hard gate)                    : {MEETS_COHERENCE_KPI} "
      f"(P(Severe->Severe)={P_SEVERE_SEVERE:.3f} vs P(Low->Severe)={P_LOW_SEVERE:.3f})")
print(f"Real transition pairs observed (HOLDOUT)    : {N_TRANSITION_PAIRS:,}")
print(f"Escalation-validity default rate (escalated vs not): "
      f"{_default_rate_escalated:.3f} vs {_default_rate_not_escalated:.3f}")
if "p_value" in P6_STRATIFICATION_RESULT and P6_STRATIFICATION_RESULT["p_value"] is not None:
    print(f"Problem 6 stratification (exploratory)       : p={P6_STRATIFICATION_RESULT['p_value']:.4f}")
print(
    "\nNext: 48_roll_rate_modeling_validation_deployment.ipynb -- deterministically reproduces this "
    "notebook's entire pipeline from scratch, bootstraps confidence intervals on the transition "
    "probabilities and the monotonicity/coherence relationships, and makes the final honest "
    "RECOMMENDED/NOT RECOMMENDED call."
)
